# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [ ]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [ ]:
import math
import sys


def exp_limit_approximation(x_value: float, max_steps: int = 128) -> tuple[float, int]:
    if not math.isfinite(x_value):
        raise ValueError("x deve ser um número finito.")

    n_value = max(1, math.ceil(abs(x_value)) + 1)
    previous_approximation = None

    for step in range(max_steps):
        approximation = math.exp(
            n_value * math.log1p(x_value / n_value)
        )

        if previous_approximation is not None and math.isclose(
            approximation,
            previous_approximation,
            rel_tol=sys.float_info.epsilon,
            abs_tol=0.0,
        ):
            return approximation, n_value

        previous_approximation = approximation
        n_value *= 2

    raise RuntimeError("A aproximação não convergiu dentro do limite de etapas.")


for x_value in (1.0, 5.0, -2.0, -5.0):
    approximation, n_value = exp_limit_approximation(x_value)
    assert math.isclose(approximation, math.exp(x_value), rel_tol=1e-14)
    print(
        f"x={x_value:+g}: aproximação={approximation:.16g}, "
        f"math.exp={math.exp(x_value):.16g}, n={n_value}"
    )

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [ ]:
import math


def exp_scaled(x_value: float, theta: float = 0.5) -> tuple[float, int, float]:
    if not math.isfinite(x_value):
        raise ValueError("x deve ser um número finito.")
    if not math.isfinite(theta) or theta <= 0:
        raise ValueError("theta deve ser um número finito e positivo.")
    if x_value == 0:
        return 1.0, 0, 0.0

    if abs(x_value) <= theta:
        k = 0
    else:
        k = math.ceil(math.log2(abs(x_value)) - math.log2(theta))

    m = math.ldexp(x_value, -k)
    approximation = exp_series(m)[0]

    squarings = k
    while k > 0:
        approximation *= approximation
        k -= 1

    return approximation, squarings, m


for x_value in (1.0, 20.0, -20.0, 100.0, -100.0):
    approximation, squarings, reduced_x = exp_scaled(x_value)
    assert math.isclose(approximation, math.exp(x_value), rel_tol=1e-12)
    print(
        f"x={x_value:+g}: aproximação={approximation:.16g}, "
        f"math.exp={math.exp(x_value):.16g}, "
        f"m={reduced_x:.6g}, k={squarings}"
    )

## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [ ]:
import math
import sys


def cos_series(x_value: float) -> tuple[float, int]:
    if not math.isfinite(x_value):
        raise ValueError("x deve ser um número finito.")

    epsilon = sys.float_info.epsilon
    total = 1.0
    term = 1.0
    correction = 0.0
    n = 0

    while n < 1000:
        term *= -(x_value * x_value) / ((2 * n + 1) * (2 * n + 2))
        n += 1

        adjusted_term = term - correction
        updated_total = total + adjusted_term
        correction = (updated_total - total) - adjusted_term
        total = updated_total

        if abs(term) <= epsilon * max(1.0, abs(total)):
            return total, n

    raise RuntimeError("A série não convergiu dentro do limite de iterações.")


sample_count = 200
x_values = [
    -20.0 + 40.0 * index / (sample_count - 1)
    for index in range(sample_count)
]
relative_errors = []
maximum_absolute_error = 0.0

for x_value in x_values:
    approximation, terms_used = cos_series(x_value)
    reference = math.cos(x_value)
    absolute_error = abs(approximation - reference)
    maximum_absolute_error = max(maximum_absolute_error, absolute_error)

    if reference != 0.0:
        relative_errors.append((absolute_error / abs(reference), x_value))

maximum_relative_error, worst_x = max(relative_errors)
print(f"Pontos comparados: {sample_count}")
print(f"Maior erro relativo: {maximum_relative_error:.6e} em x={worst_x:.6f}")
print(f"Maior erro absoluto: {maximum_absolute_error:.6e}")

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [ ]:
import math
import sys


def remainder_upper_bound(x_abs: float, first_index: int) -> float:
    if x_abs == 0.0:
        return 0.0

    log_first_term = (
        first_index * math.log(x_abs) - math.lgamma(first_index + 1)
    )
    if log_first_term > math.log(sys.float_info.max):
        return math.inf

    term = math.exp(log_first_term)
    if term == 0.0:
        return 0.0

    tail_sum = term
    index = first_index
    steps = 0

    while steps < 100_000:
        ratio = x_abs / (index + 1)
        if ratio < 1.0:
            remaining_bound = term * ratio / (1.0 - ratio)
            if remaining_bound <= sys.float_info.epsilon * tail_sum:
                return tail_sum + remaining_bound

        index += 1
        term *= x_abs / index
        tail_sum += term
        steps += 1

        if term == 0.0:
            return tail_sum

    raise RuntimeError("A soma da cauda não convergiu dentro do limite de iterações.")


def find_minimum_n(x_value: float, tau: float) -> tuple[int, float]:
    if not math.isfinite(x_value):
        raise ValueError("x deve ser um número finito.")
    if not math.isfinite(tau) or tau <= 0.0:
        raise ValueError("tau deve ser um número finito e positivo.")

    x_abs = abs(x_value)
    n_value = 0

    while n_value < 100_000:
        remainder_bound = remainder_upper_bound(x_abs, n_value + 1)
        if remainder_bound < tau:
            return n_value, remainder_bound
        n_value += 1

    raise RuntimeError("N não foi encontrado dentro do limite de busca.")


for x_value, tau in ((2.0, 1e-6), (5.0, 1e-8)):
    n_value, remainder_bound = find_minimum_n(x_value, tau)
    assert remainder_bound < tau
    if n_value > 0:
        previous_bound = remainder_upper_bound(abs(x_value), n_value)
        assert previous_bound >= tau
    print(
        f"x={x_value:g}, tau={tau:.1e}: N={n_value}, "
        f"cota da cauda={remainder_bound:.3e}"
    )

## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [ ]:
import math
import sys
from decimal import Decimal, localcontext


def exp_series_float64(x_value: float) -> float:
    epsilon = sys.float_info.epsilon
    total = 1.0
    term = 1.0

    for n in range(1, 10_001):
        term *= x_value / n
        total += term
        if abs(term) <= epsilon * abs(total):
            return total

    raise RuntimeError("A série não convergiu dentro do limite de iterações.")


with localcontext() as context:
    context.prec = 80

    print(f"Precisão do Decimal: {context.prec} dígitos")
    print("Comparação de e^x:")

    for x_value in (20, 40, 50):
        reference = Decimal(x_value).exp()
        float64_result = exp_series_float64(float(x_value))
        relative_error = (
            abs(Decimal.from_float(float64_result) - reference) / abs(reference)
        )
        relative_error_float = float(relative_error)
        significant_digits = (
            -math.log10(relative_error_float)
            if relative_error_float > 0.0
            else math.inf
        )

        print(
            f"x={x_value}: Decimal={reference:.20E}, "
            f"float64={float64_result:.16e}, "
            f"erro relativo={relative_error:.3E}, "
            f"dígitos significativos≈{significant_digits:.2f}"
        )

    overflow_limit = math.log(sys.float_info.max)
    print(f"\nln(maior float64 finito) ≈ {overflow_limit:.12f}")

    for x_probe in (overflow_limit, 710.0):
        try:
            print(f"math.exp({x_probe:.12f}) = {math.exp(x_probe):.6e}")
        except OverflowError:
            print(f"math.exp({x_probe:.12f}) excede o limite de float64")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```